In [22]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
import os

model_one=init_chat_model(
    model_name="deepseek-chat",
    model_provider="openai",
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5"
)

d:\code\langchain\03-Agentic-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 71/71 [00:00<00:00, 5083.22it/s]


In [23]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="my_documents",
    embedding_function=embeddings,
    persist_directory="./db/chroma_db",
)


In [6]:

import os
from langchain_mineru import MinerULoader
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path

files=[str(p) for p in Path("./resources").rglob("*")
       if p.suffix.lower() in [".pdf",".doc",".docx",".txt"]]

loader=MinerULoader(
    files,
    mode="flash",
    timeout=300,
)
docs=loader.load()

output_dir=Path("./resources/output")
output_dir.mkdir(parents=True,exist_ok=True)

for doc in docs:
    source_file=Path(doc.metadata["source"]).stem
    output_file=output_dir/f"{source_file}.md"
    print(doc.page_content)
    output_file.write_text(
        data=doc.page_content,
        encoding="utf-8",
    )


KeyboardInterrupt: 

In [24]:

from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on=[
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

spliter=MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,
)

with open("./resources/output/note.md","r",encoding="utf-8") as f:
    doc=f.read()

chunks=spliter.split_text(doc)


In [25]:
vectorstore.add_documents(chunks)
retriever=vectorstore.as_retriever()

In [26]:


import os
from pathlib import Path
import pkuseg
import bm25s

seg=pkuseg.pkuseg()

def create_bm25_index(documents,dir_str="./db/bm25_index",k1=1.5,b=0.75):
    dir=Path(dir_str)
    dir.mkdir(parents=True,exist_ok=True)

    metadata_corpus=[
        {
            "id":doc.metadata.get("source",str(i)),
            "content":doc.page_content,
        }
        for i,doc in enumerate(documents)
    ]
    #用来判断用的，现在没写判断逻辑
    file=dir / "data.csc.index.npy"

    if(file.exists()):
        print(f"加载已有索引:{dir}")
        bm_retriever=bm25s.BM25.load(
            str(dir),
            load_corpus=True,
        )
    else:
        bm_retriever=bm25s.BM25(
            k1=k1,
            b=b,
            corpus=metadata_corpus
        )

        corpus=[
            seg.cut(item["content"])
            for item in metadata_corpus
        ]

        bm_retriever.index(corpus=corpus)

        bm_retriever.save(str(dir),metadata_corpus)

    return bm_retriever,metadata_corpus




In [27]:
bm_retriever,metadata_corpus=create_bm25_index(chunks)

加载已有索引:db\bm25_index


In [28]:
from typing import Tuple,List,Dict

def bm25_search(query:str,top_k:int=3)->List[Tuple[Dict,float]]:
    query_one=[seg.cut(query)]
    results,scores=bm_retriever.retrieve(query_tokens=query_one,k=top_k)
    return [(results[0][i],scores[0][i]) for i in range(results.shape[1])]

'''query = "给教师的建议是谁写的"
# 调用工具
ranked_docs = bm25_search(query, top_k = 1)

for i, (doc, score) in enumerate(ranked_docs):
    print(f"======================Rank {i+1} (score: {score:.2f})=================")
    print(f"doc: {doc}")'''


def bm25_search_batch(query_list:List[str],top_k:int=3)->List[List[Tuple[Dict,float]]]:
    query_list_tokens=[seg.cut(query) for query in query_list]
    results,scores=bm_retriever.retrieve(query_tokens=query_list_tokens,k=top_k)
    return [
        [(results[b][i],scores[b][i])for i in range(results.shape[1])]
        for b in range(results.shape[0])
    ]



In [20]:
import torch
from sentence_transformers import CrossEncoder

# 首次运行会从镜像源自动下载模型到本地缓存（默认路径：~/.cache/huggingface/hub/）
# 后续运行直接读取缓存，无需重复下载
model = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

d:\code\langchain\03-Agentic-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo1\.cache\huggingface\hub\models--Qwen--Qwen3-Reranker-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\code\langchain\03-Agentic-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarn

In [29]:

from typing import List,Dict

def cross_encoder_rerank(query:str,docs:List[Dict],top_k=3):
    scores=model.predict([(query,doc["content"]) for doc in docs])
    for i,s in enumerate(scores):
        docs[i]["score"]=s
    sorted_docs=sorted(docs,key=lambda x:x["score"],reverse=True)
    positive_docs = [doc for doc in sorted_docs if doc["score"] > 0]
    return positive_docs[0:min(len(positive_docs),top_k)]
    

In [30]:
from typing import List,Dict

def reciprocal_rank_fusion(ranked_list:List[List[Dict]],k:int=60):
    """
    ranked_lists: List[List[Dict]]
                  每个检索器返回的文档dict列表，包含id和content，按排名从高到低。
                  注意：这里不需要score，在 RRF 中只使用排名位置。
    """
    results_list={}
    rrf_scores={}
    for rank,one_list in enumerate(ranked_list,1):
        for doc in one_list:
            doc_id=doc["id"]
            rrf_scores[doc_id]=rrf_scores.get(doc_id,0)+1/(k+rank)
            results_list[doc_id]=doc
    sorted_rrf_scores=sorted(rrf_scores.items(),key=lambda x:x[1],reverse=True)
    return [results_list[doc_id] for doc_id,score in sorted_rrf_scores]



In [31]:
import time
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search_knowledge_base(query:str):
    '''
    搜索知识库，获取关键知识，需要查找资料时调用
    '''
    start=time.perf_counter()
    vector_results=vectorstore.similarity_search_with_score(query=query)
    end=time.perf_counter()
    print(f"向量数据库检索耗时:{end-start}")

    start=time.perf_counter()
    bm25_results=bm25_search(query=query,top_k=1)
    end=time.perf_counter()
    print(f"精确检索耗时:{end-start}")

    if not vector_results or not bm25_results:
        return "未找到相关文档"

    vec_doc=[
        {
            "id":doc.id,
            "content":doc.page_content,
        }
        for doc,_ in vector_results
    ]
    bm_doc=[
        {
            "id":doc["id"],
            "content":doc["content"],
        }
        for doc,_ in bm25_results
    ]

    rrf_docs=reciprocal_rank_fusion([vec_doc,bm_doc])

    start=time.perf_counter()
    final_docs=cross_encoder_rerank(query=query,docs=rrf_docs)
    end=time.perf_counter()
    print(f"cross-encoder精排:{end-start}")

    docs_content="\n\n".join(doc["content"] for doc in final_docs)
    print(f"检索问题:{query}")
    print(f"相关文档:{docs_content}")
    return docs_content

In [32]:
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent
rag_agent=create_agent(
    model="deepseek-chat",
    tools=[search_knowledge_base],
    system_prompt='''
    你是一个专业的教师资格证考试辅导专家。您的主要职责是帮助用户解决有关教资考试的相关问题。产品说明:
    1。如果用户问了一个你不确定的问题，或者涉及教资专业知识的问题，你必须使用`search_knowledge_base`工具来查阅相关文档。
    2. 在引用文档时，要清楚地总结包括内容中的相关上下文。
    3. 如果获取文档失败，请告诉用户，并以您最好的专家理解继续进行。
    在回答用户关于教资考试知识的问题之前，您必须查阅工具以获取最新信息。你的回答应该清晰、简洁、准确。不要有过多解释除非用户询问。
    ''',
)
print("rag_agent初始化完成")

rag_agent初始化完成


In [40]:
from langchain.messages import HumanMessage,AIMessage

response=rag_agent.stream(
    {
        "messages":[HumanMessage(content="你好")]
    },
    stream_mode="messages"
)
for chunk,metadata in response:
    if isinstance(chunk,AIMessage) and chunk.content:
        print(chunk.content,flush=True)

你好
！
我是
教
资
考试
辅导
专家
，
有什么
关于
教师资格
证
考试
的问题
可以
帮你
解答
？
比如
报考
条件
、
考试
科目
、
备考
方法
等
，
尽管
问我
。


In [41]:
from langchain.messages import AIMessage
# 检索
response = rag_agent.stream(
    {"messages": [{"role": "user", "content": "谁的教学原则是因材施教、启发诱导、学思结合、谦虚笃实？，这个人是谁"}]},
    stream_mode="messages"
)

for chunk, metadata in response:
    print(chunk.content, end="")

向量数据库检索耗时:0.060834199999590055


精确检索耗时:0.0036354999992909143


cross-encoder精排:0.383128299999953
检索问题:因材施教 启发诱导 学思结合 谦虚笃实 教学原则 谁
相关文档:### **（三）孔子及《论语》主要思想**  
\- 教育作用：庶、富、教；性相近，习相远。  
\- 教育对象：“有教无类”，教育民主思想。  
\- 教育目的：以完善人格为教育的首要目的，培养士和君子。  
\- 教学内容：文、行、忠、义。  
\- 教学过程：学、思、习、行。  
\- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。  
\- 教师观：其身正，不令而行；其身不正，虽令不从。

### **（三）孔子及《论语》主要思想**  
\- 教育作用：庶、富、教；性相近，习相远。  
\- 教育对象：“有教无类”，教育民主思想。  
\- 教育目的：以完善人格为教育的首要目的，培养士和君子。  
\- 教学内容：文、行、忠、义。  
\- 教学过程：学、思、习、行。  
\- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。  
\- 教师观：其身正，不令而行；其身不正，虽令不从。

第一章 教育概述  
第一节 中外教育家及其教育思想  
（一）古代希腊教育家  
\- 苏格拉底：产婆术，通过问答引导学生自己发现真理。  
\- 柏拉图：主张教育应培养哲学王，强调理性与德性的统一。  
\- 亚里士多德：提出德智体和谐发展，重视习惯养成与自由教育。  
（二）近代西方教育家  
\- 夸美纽斯：提出班级授课制，著有《大教学论》，主张“把一切事物教给一切人”。  
\- 卢梭：自然主义教育，强调儿童天性，著有《爱弥儿》。  
\- 杜威：实用主义教育，主张“从做中学”，教育即生活、教育即生长。  
（三）孔子及《论语》主要思想  
\- 教育作用：庶、富、教；性相近，习相远。  
\- 教育对象：“有教无类”，教育民主思想。  
\- 教育目的：以完善人格为教育的首要目的，培养士和君子。  
\- 教学内容：文、行、忠、义。  
\- 教学过程：学、思、习、行。  
\- 教学原则：因材施教、启发诱导、学思结合、谦虚笃实。  
\- 教师观：其身正，不令而行；其身不正，虽令不从。  
（四）孟子主要思想  
\- 人性论：人性本善，人先天具有仁、义、礼、智四个“善端”。  
\- 教育作用：发扬善端，培养道